In [1]:
import numpy as np
import pandas as pd
import math
import os

def get_split_row(chr_file):
    with np.load(chr_file, allow_pickle=True) as data:
        assert "chr10" in data, "chr10 not found in file"
        chr10_rows = data["chr10"].shape[0]
        split_row = math.ceil(chr10_rows / 1024) * 1024
        print(f"chr10 row count: {chr10_rows}, split row count: {split_row}\n")
        return split_row

def process_single_npz(file_path, pred_key, output_prefix, split_row):
    if not os.path.exists(file_path):
        print(f"Warning: File does not exist - {file_path}, skipping processing\n")
        return None

    output_dir = "processed_data"
    os.makedirs(output_dir, exist_ok=True)
    print(f"Results will be saved to: {os.path.abspath(output_dir)}")

    print(f"Processing file: {file_path}")
    with np.load(file_path, allow_pickle=True) as data:
        assert "y_true" in data, "y_true not found in file"
        assert pred_key in data, f"{pred_key} not found in file"
        
        y_true = data["y_true"].astype(np.float32)
        y_pred = data[pred_key].astype(np.float32)
        
        assert y_true.shape == y_pred.shape, f"Shape mismatch between y_true and {pred_key}"
        print(f"Data shape: {y_true.shape}, total row count: {y_true.shape[0]}")
        
        actual_split = min(split_row, y_true.shape[0])
        y_true1, y_true2 = y_true[:actual_split], y_true[actual_split:]
        y_pred1, y_pred2 = y_pred[:actual_split], y_pred[actual_split:]
        print(f"Split completed: Part 1 has {y_true1.shape[0]} rows, Part 2 has {y_true2.shape[0]} rows")

    def save(matrix, part, is_true):
        name = "y_true" if is_true else pred_key
        file_name = f"{output_prefix}{name}_part{part}.csv"
        path = os.path.join(output_dir, file_name)
        
        if matrix.size == 0:
            print(f"Warning: {name} Part {part} is empty, not saving\n")
            return
        
        pd.DataFrame(
            matrix,
            index=[f"row_{i}" for i in range(matrix.shape[0])],
            columns=[f"col_{i}" for i in range(matrix.shape[1])]
        ).to_csv(path, index=True, header=True, na_rep="NaN", encoding="utf-8")
        print(f"{name} Part {part} saved to: {path}")

    save(y_true1, 1, is_true=True)
    save(y_true2, 2, is_true=True)
    save(y_pred1, 1, is_true=False)
    save(y_pred2, 2, is_true=False)

    print(f"\nStatistics:")
    print(f"y_true Part 1: {y_true1.shape} NaN values: {np.isnan(y_true1).sum()}")
    print(f"y_true Part 2: {y_true2.shape} NaN values: {np.isnan(y_true2).sum() if y_true2.size else 0}")
    print(f"{pred_key} Part 1: {y_pred1.shape} NaN values: {np.isnan(y_pred1).sum()}")
    print(f"{pred_key} Part 2: {y_pred2.shape} NaN values: {np.isnan(y_pred2).sum() if y_pred2.size else 0}\n")

    print(f"Preview of first 5 rows and 5 columns:")
    print(pd.DataFrame(y_true1[:5, :5]).round(4), "\n" + "-"*50 + "\n")
    
    return {
        "y_true1": y_true1,
        "y_true2": y_true2,
        "y_pred1": y_pred1,
        "y_pred2": y_pred2
    }

def main():
    chr_npz = "/home/methyeval/data/lianght/cpg-transformer-main/GSE2/y_batch_30.npz"
    mamba_npz = "MambaCpG.npz"
    transformer_npz = "CpGTransformer.npz"
    
    mamba_prefix = "mamba_"
    transformer_prefix = "transformer_"
    
    split_row = get_split_row(chr_npz)
    
    model_results = {}
    model_results["mamba"] = process_single_npz(
        file_path=mamba_npz,
        pred_key="y_pred_prob",
        output_prefix=mamba_prefix,
        split_row=split_row
    )
    
    model_results["transformer"] = process_single_npz(
        file_path=transformer_npz,
        pred_key="y_pred_prob",
        output_prefix=transformer_prefix,
        split_row=split_row
    )
    
    print("All files processed!")
    return model_results

if __name__ == "__main__":
    results = main()
    
    if results["mamba"]:
        mamba_y_true1 = results["mamba"]["y_true1"]
        mamba_y_true2 = results["mamba"]["y_true2"]
        print(f"Mamba retained y_true1 shape: {mamba_y_true1.shape}")
    
    if results["transformer"]:
        transformer_y_true1 = results["transformer"]["y_true1"]
        transformer_y_true2 = results["transformer"]["y_true2"]
        print(f"Transformer retained y_true1 shape: {transformer_y_true1.shape}")
    

chr10 row count: 207613, split row count: 207872

Results will be saved to: /data/cabins/methyeval/lianght/cpg-transformer-main/merge_pott/processed_data
Processing file: MambaCpG.npz
Data shape: (400384, 30), total row count: 400384
Split completed: Part 1 has 207872 rows, Part 2 has 192512 rows
y_true Part 1 saved to: processed_data/mamba_y_true_part1.csv
y_true Part 2 saved to: processed_data/mamba_y_true_part2.csv
y_pred_prob Part 1 saved to: processed_data/mamba_y_pred_prob_part1.csv
y_pred_prob Part 2 saved to: processed_data/mamba_y_pred_prob_part2.csv

Statistics:
y_true Part 1: (207872, 30) NaN values: 6146621
y_true Part 2: (192512, 30) NaN values: 5693640
y_pred_prob Part 1: (207872, 30) NaN values: 6146621
y_pred_prob Part 2: (192512, 30) NaN values: 5693640

Preview of first 5 rows and 5 columns:
    0   1   2   3   4
0 NaN NaN NaN NaN NaN
1 NaN NaN NaN NaN NaN
2 NaN NaN NaN NaN NaN
3 NaN NaN NaN NaN NaN
4 NaN NaN NaN NaN NaN 
----------------------------------------------

In [2]:
import numpy as np
import pandas as pd
import os

INPUT_NPZ = "graphCpG.npz"
OUTPUT_PART1 = "graphCpG_part1.csv"
OUTPUT_PART2 = "graphCpG_part2.csv"
M_DIFF_THRESHOLD = 100000
REMOVE_ROWS = 10  # Remove rows 0-9 (m=0-10)
output_dir = "processed_data"
os.makedirs(output_dir, exist_ok=True)
OUTPUT_PART1 = os.path.join(output_dir, OUTPUT_PART1)
OUTPUT_PART2 = os.path.join(output_dir, OUTPUT_PART2)

# Step 1: Load data
print(f"Loading {INPUT_NPZ}...")
with np.load(INPUT_NPZ, allow_pickle=True) as data:
    y = data["y_pred_sigmoid"]
    m = data["m_global"]
    n = data["n_global"]

assert len(y) == len(m) == len(n), "Data length mismatch"
print(f"Total records: {len(y)}, m range: {m.min()}~{m.max()}")

# Step 2: Split data into two parts first
print(f"\nFinding split point (abs m difference > {M_DIFF_THRESHOLD})...")
m_diffs = np.diff(m)
large_diff_idx = np.where(np.abs(m_diffs) > M_DIFF_THRESHOLD)[0]

if len(large_diff_idx) > 0:
    split_idx = large_diff_idx[0] + 1
    print(f"Split at index {split_idx} (abs diff: {np.abs(m_diffs[large_diff_idx[0]])})")
else:
    split_idx = len(y) // 2
    print(f"No large difference found, split at midpoint: {split_idx}")

# Split into two parts
part1 = {"y": y[:split_idx], "m": m[:split_idx], "n": n[:split_idx]}
part2 = {"y": y[split_idx:], "m": m[split_idx:], "n": n[split_idx:]}

print(f"\nSplit results:")
print(f"Part1: {len(part1['y'])} records, m range: {part1['m'].min()}~{part1['m'].max()}")
print(f"Part2: {len(part2['y'])} records, m range: {part2['m'].min()}~{part2['m'].max()}")

# Step 3: Build matrix for each part and export to CSV
def build_matrix(data, output_path):
    if len(data["y"]) == 0:
        print(f"Warning: No data for {output_path}")
        return None
    
    # Get full m range for matrix dimensions
    all_m = data["m"]
    max_m = all_m.max()
    max_n = data["n"].max() if len(data["n"]) > 0 else 0
    
    # Create full matrix (rows = m values, columns = n values)
    full_matrix = np.full((max_m + 1, max_n + 1), np.nan, dtype=np.float32)
    
    # Fill matrix with valid data (m >= 11)
    valid_mask = all_m >= 11
    valid_m = all_m[valid_mask]
    valid_n = data["n"][valid_mask]
    valid_y = data["y"][valid_mask]
    for m_val, n_val, y_val in zip(valid_m, valid_n, valid_y):
        if m_val <= max_m and n_val <= max_n:
            full_matrix[m_val, n_val] = y_val
    
    # Remove first 10 rows (m=0 to m=9)
    if full_matrix.shape[0] > REMOVE_ROWS:
        matrix_after_remove = full_matrix[REMOVE_ROWS:]  # Keep rows 10 and beyond
        print(f"  Removed first {REMOVE_ROWS} rows (m=0-9) from {output_path}")
    else:
        matrix_after_remove = full_matrix
        print(f"  Warning: {output_path} matrix too small to remove {REMOVE_ROWS} rows")
    
    # Adjust rows to make m=11 correspond to row_0
    # m=11 was at row 11 in full matrix → after removing 10 rows, it's at row 1 → need to shift 1 more
    shift = 1  # Because m=11 is at row 1 after removing 10 rows (11-10=1)
    if matrix_after_remove.shape[0] > shift:
        final_matrix = matrix_after_remove[shift:]  # Now m=11 is at row 0
    else:
        final_matrix = matrix_after_remove
        print(f"  Warning: {output_path} matrix too small for final shift")
    
    # Create DataFrame with row_0, row_1...
    row_indices = [f"row_{i}" for i in range(final_matrix.shape[0])]
    col_indices = [f"col_{i}" for i in range(final_matrix.shape[1])]
    df = pd.DataFrame(final_matrix, index=row_indices, columns=col_indices)
    
    # Save to CSV
    df.to_csv(output_path, index=True, header=True, na_rep="NaN")
    print(f"  Saved {output_path} (shape: {final_matrix.shape})")
    return df

# Step 4: Process both parts
print("\nBuilding matrices...")
df1 = build_matrix(part1, OUTPUT_PART1)
df2 = build_matrix(part2, OUTPUT_PART2)

# Step 5: Verify results
print("\n=== Verification ===")
if df1 is not None:
    print(f"Part1 matrix shape: {df1.shape}")
    print(f"Part1 row_0 (should correspond to m=11) first 5 columns:\n{df1.iloc[:1, :5].round(4) if not df1.empty else 'Empty'}")

if df2 is not None:
    print(f"Part2 matrix shape: {df2.shape}")
    print(f"Part2 row_0 (should correspond to m=11) first 5 columns:\n{df2.iloc[:1, :5].round(4) if not df2.empty else 'Empty'}")
    

Loading graphCpG.npz...
Total records: 684424, m range: 11~207623

Finding split point (abs m difference > 100000)...
Split at index 357765 (abs diff: 207612)

Split results:
Part1: 357765 records, m range: 11~207623
Part2: 326659 records, m range: 11~192406

Building matrices...
  Removed first 10 rows (m=0-9) from processed_data/graphCpG_part1.csv
  Saved processed_data/graphCpG_part1.csv (shape: (207613, 30))
  Removed first 10 rows (m=0-9) from processed_data/graphCpG_part2.csv
  Saved processed_data/graphCpG_part2.csv (shape: (192396, 30))

=== Verification ===
Part1 matrix shape: (207613, 30)
Part1 row_0 (should correspond to m=11) first 5 columns:
       col_0  col_1  col_2  col_3  col_4
row_0    NaN    NaN    NaN    NaN    NaN
Part2 matrix shape: (192396, 30)
Part2 row_0 (should correspond to m=11) first 5 columns:
       col_0  col_1  col_2  col_3  col_4
row_0    NaN    NaN    NaN    NaN    NaN


In [3]:
import os
import pandas as pd

def adjust_csv_rows(file_path, target_rows):
    df = pd.read_csv(file_path, index_col=0)
    if len(df) > target_rows:
        df.iloc[:target_rows].to_csv(file_path, index=True, header=True, na_rep="NaN", encoding="utf-8")

if __name__ == "__main__":
    target_part1 = mamba_y_true1.shape[0] - 1024
    target_part2 = mamba_y_true2.shape[0] - 1024
    data_dir = "./processed_data"
    
    for f in os.listdir(data_dir):
        if f.endswith(".csv") and ("part1" in f.lower() or "part2" in f.lower()):
            file_path = os.path.join(data_dir, f)
            target = target_part1 if "part1" in f.lower() else target_part2
            adjust_csv_rows(file_path, target)
    
    print("Processing completed")

Processing completed


In [4]:
import pandas as pd
import numpy as np
import os

def sync_nan_ignore_index(source_file, target_file, backup_suffix="_backup"):
    try:
        df_source = pd.read_csv(source_file)
        df_target = pd.read_csv(target_file)
        
        if df_source.shape[1] != df_target.shape[1]:
            raise ValueError(f"Column count mismatch: source has {df_source.shape[1]}, target has {df_target.shape[1]}")
        
        if len(df_source) < 2 or len(df_target) < 2:
            raise ValueError("Insufficient rows (need header + at least 1 data row)")
        
        source_data = df_source.iloc[0:, 1:]
        target_data = df_target.iloc[0:, 1:]
        
        if len(source_data) != len(target_data):
            raise ValueError(f"Row count mismatch: source has {len(source_data)}, target has {len(target_data)}")
        
        nan_mask = source_data.isna()
        if nan_mask.sum().sum() > 0:
            backup_file = f"{os.path.splitext(target_file)[0]}{backup_suffix}.csv"
            df_target.to_csv(backup_file, index=False, header=True, na_rep="NaN", encoding="utf-8")
            
            target_data[nan_mask] = np.nan
            df_target.iloc[0:, 1:] = target_data
            df_target.to_csv(target_file, index=False, header=True, na_rep="NaN", encoding="utf-8")
            print(f"Processed: {os.path.basename(source_file)} → {os.path.basename(target_file)} (backup created)")
        else:
            print(f"No action needed: {os.path.basename(source_file)} contains no NaNs")
    
    except Exception as e:
        print(f"Error processing {os.path.basename(source_file)}→{os.path.basename(target_file)}: {str(e)}")

if __name__ == "__main__":
    data_dir = "./processed_data"
    file_pairs = [
        (os.path.join(data_dir, "transformer_y_true_part1.csv"),os.path.join(data_dir, "graphCpG_part1.csv")),
        (os.path.join(data_dir, "transformer_y_true_part2.csv"),os.path.join(data_dir, "graphCpG_part2.csv"))
    ]
    
    for source, target in file_pairs:
        sync_nan_ignore_index(source, target)
    

Processed: transformer_y_true_part1.csv → graphCpG_part1.csv (backup created)
Processed: transformer_y_true_part2.csv → graphCpG_part2.csv (backup created)


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, accuracy_score, matthews_corrcoef,
                             confusion_matrix, f1_score, roc_curve)
from lightgbm import LGBMClassifier
import os
import joblib
from scipy import stats
from sklearn.model_selection import StratifiedKFold, GroupKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

def custom_find_peaks(data, height=None, distance=1):
    if len(data) < 3:
        return np.array([], dtype=int), {'peak_heights': np.array([])}
    
    peaks = []
    for i in range(1, len(data)-1):
        if data[i] > data[i-1] and data[i] > data[i+1]:
            peaks.append(i)
    peaks = np.array(peaks, dtype=int)
    
    if height is not None:
        peak_heights = data[peaks]
        valid_mask = peak_heights >= height
        peaks = peaks[valid_mask]
    
    if distance > 1 and len(peaks) > 1:
        filtered_peaks = [peaks[0]]
        for peak in peaks[1:]:
            if peak - filtered_peaks[-1] >= distance:
                filtered_peaks.append(peak)
        peaks = np.array(filtered_peaks, dtype=int)
    
    return peaks, {'peak_heights': data[peaks] if len(peaks) > 0 else np.array([])}

file_paths_train = {
    "graph": os.path.join("processed_data", "graphCpG_part2.csv"),
    "mamba": os.path.join("processed_data", "mamba_y_pred_prob_part2.csv"), 
    "transformer": os.path.join("processed_data", "transformer_y_pred_prob_part2.csv"),
    "true": os.path.join("processed_data", "transformer_y_true_part2.csv")
}

file_paths_test = {
    "graph": os.path.join("processed_data", "graphCpG_part1.csv"),
    "mamba": os.path.join("processed_data", "mamba_y_pred_prob_part1.csv"),
    "transformer": os.path.join("processed_data", "transformer_y_pred_prob_part1.csv"),
    "true": os.path.join("processed_data", "transformer_y_true_part1.csv")
}

output_dir = "./meta_results"
os.makedirs(output_dir, exist_ok=True)

ENSEMBLE_PARAMS = {
    "boosting_type": "gbdt",
    "n_estimators": 300,        
    "learning_rate": 0.025,     
    "max_depth": 8,             
    "num_leaves": 40,           
    "min_child_samples": 25,    
    "subsample": 0.85,          
    "colsample_bytree": 0.85,   
    "reg_alpha": 0.2,           
    "reg_lambda": 0.2,          
    "random_state": 42,
    "verbose": 50,
    "n_jobs": -1
}

THRESHOLD = 0.5
WINDOW_SIZE = 10

class AdvancedNeighborhoodFeatureExtractor:
    def __init__(self, window_size=WINDOW_SIZE):
        self.window_size = window_size
    
    def extract_features_for_dataset(self, data_dict):
        graph_data = data_dict["graph"]
        mamba_data = data_dict["mamba"]
        transformer_data = data_dict["transformer"]
        true_data = data_dict["true"]
        
        n_rows, n_cols = graph_data.shape
        all_features = []
        
        for col_idx in range(n_cols):
            graph_col = graph_data.iloc[:, col_idx].values
            mamba_col = mamba_data.iloc[:, col_idx].values
            transformer_col = transformer_data.iloc[:, col_idx].values
            true_col = true_data.iloc[:, col_idx].values
            
            col_features = self._extract_features_for_column(
                graph_col, mamba_col, transformer_col, true_col, col_idx
            )
            all_features.extend(col_features)
        
        features_df = pd.DataFrame(all_features)
        return features_df
    
    def _extract_features_for_column(self, graph_col, mamba_col, transformer_col, true_col, col_idx):
        features = []
        n_positions = len(graph_col)
        
        for pos in range(n_positions):
            if (np.isnan(graph_col[pos]) or np.isnan(mamba_col[pos]) or 
                np.isnan(transformer_col[pos]) or np.isnan(true_col[pos])):
                continue
            
            graph_val = graph_col[pos]
            mamba_val = mamba_col[pos]
            transformer_val = transformer_col[pos]
            
            weighted_avg = 0.25 * graph_val + 0.4 * mamba_val + 0.35 * transformer_val
            max_val = max(graph_val, mamba_val, transformer_val)
            min_val = min(graph_val, mamba_val, transformer_val)
            range_val = max_val - min_val
            std_val = np.std([graph_val, mamba_val, transformer_val])
            
            base_features = {
                'graph_current': graph_val,
                'mamba_current': mamba_val,
                'transformer_current': transformer_val,
                'weighted_avg': weighted_avg,  
                'max_score': max_val,          
                'min_score': min_val,          
                'score_range': range_val,      
                'score_std': std_val,          
                'avg_score': (graph_val + mamba_val + transformer_val) / 3,  
                'y_true': int(true_col[pos]),
                'col_idx': col_idx,
                'pos_idx': pos
            }
            
            start_idx = max(0, pos - self.window_size)
            end_idx = min(n_positions, pos + self.window_size + 1)
            
            graph_window = graph_col[start_idx:end_idx]
            mamba_window = mamba_col[start_idx:end_idx]
            transformer_window = transformer_col[start_idx:end_idx]
            
            graph_valid = graph_window[~np.isnan(graph_window)]
            mamba_valid = mamba_window[~np.isnan(mamba_window)]
            transformer_valid = transformer_window[~np.isnan(transformer_window)]
            
            advanced_features = {}
            advanced_features.update(self._extract_statistical_features(graph_valid, 'graph'))
            advanced_features.update(self._extract_statistical_features(mamba_valid, 'mamba'))
            advanced_features.update(self._extract_statistical_features(transformer_valid, 'transformer'))
            advanced_features.update(self._extract_shape_features(graph_valid, 'graph'))
            advanced_features.update(self._extract_shape_features(mamba_valid, 'mamba'))
            advanced_features.update(self._extract_shape_features(transformer_valid, 'transformer'))
            advanced_features.update(self._extract_positional_features(
                graph_col, mamba_col, transformer_col, pos, start_idx, end_idx
            ))
            if len(graph_valid) > 2 and len(mamba_valid) > 2:
                advanced_features.update(self._extract_correlation_features(
                    graph_valid, mamba_valid, transformer_valid
                ))
            
            all_sample_features = {**base_features, **advanced_features}
            features.append(all_sample_features)
        
        return features
    
    def _extract_statistical_features(self, data, prefix):
        features = {}
        if len(data) > 0:
            features.update({
                f'{prefix}_neighbor_mean': np.mean(data),
                f'{prefix}_neighbor_std': np.std(data) if len(data) > 1 else 0,
                f'{prefix}_neighbor_median': np.median(data),
                f'{prefix}_neighbor_range': np.ptp(data) if len(data) > 1 else 0,
                f'{prefix}_neighbor_iqr': stats.iqr(data) if len(data) > 4 else 0,
                f'{prefix}_neighbor_skew': stats.skew(data) if len(data) > 2 else 0,
                f'{prefix}_neighbor_kurtosis': stats.kurtosis(data) if len(data) > 3 else 0,
                f'{prefix}_neighbor_valid_count': len(data)
            })
            quantiles = np.quantile(data, [0.1, 0.25, 0.75, 0.9])
            for i, q in enumerate([10, 25, 75, 90]):
                features[f'{prefix}_neighbor_q{q}'] = quantiles[i]
        else:
            for stat in ['mean', 'std', 'median', 'range', 'iqr', 'skew', 'kurtosis', 'valid_count']:
                features[f'{prefix}_neighbor_{stat}'] = 0
            for q in [10, 25, 75, 90]:
                features[f'{prefix}_neighbor_q{q}'] = 0
        return features
    
    def _extract_shape_features(self, data, prefix):
        features = {}
        if len(data) > 2:
            try:
                x = np.arange(len(data))
                slope, intercept, r_value, p_value, std_err = stats.linregress(x, data)
                features.update({
                    f'{prefix}_neighbor_trend_slope': slope,
                    f'{prefix}_neighbor_trend_r2': r_value**2,
                    f'{prefix}_neighbor_trend_pvalue': p_value
                })
                if len(data) > 4:
                    peaks, properties = custom_find_peaks(data, height=np.mean(data), distance=2)
                    features[f'{prefix}_neighbor_peak_count'] = len(peaks)
                    features[f'{prefix}_neighbor_peak_mean_height'] = np.mean(properties['peak_heights']) if len(peaks) > 0 else 0
                else:
                    features[f'{prefix}_neighbor_peak_count'] = 0
                    features[f'{prefix}_neighbor_peak_mean_height'] = 0
            except:
                features.update({
                    f'{prefix}_neighbor_trend_slope': 0,
                    f'{prefix}_neighbor_trend_r2': 0,
                    f'{prefix}_neighbor_trend_pvalue': 1,
                    f'{prefix}_neighbor_peak_count': 0,
                    f'{prefix}_neighbor_peak_mean_height': 0
                })
        else:
            features.update({
                f'{prefix}_neighbor_trend_slope': 0,
                f'{prefix}_neighbor_trend_r2': 0,
                f'{prefix}_neighbor_trend_pvalue': 1,
                f'{prefix}_neighbor_peak_count': 0,
                f'{prefix}_neighbor_peak_mean_height': 0
            })
        return features
    
    def _extract_positional_features(self, graph_col, mamba_col, transformer_col, pos, start_idx, end_idx):
        features = {}
        window_center = (start_idx + end_idx - 1) / 2
        relative_pos = pos - window_center
        features['position_relative_to_center'] = relative_pos
        features['distance_to_start'] = pos - start_idx
        features['distance_to_end'] = end_idx - 1 - pos
        graph_window = graph_col[start_idx:end_idx]
        mamba_window = mamba_col[start_idx:end_idx]
        transformer_window = transformer_col[start_idx:end_idx]
        graph_valid = graph_window[~np.isnan(graph_window)]
        mamba_valid = mamba_window[~np.isnan(mamba_window)]
        transformer_valid = transformer_window[~np.isnan(transformer_window)]
        if len(graph_valid) > 0:
            current_graph = graph_col[pos]
            features['graph_current_rank'] = np.mean(current_graph > graph_valid) if len(graph_valid) > 0 else 0.5
            features['graph_current_zscore'] = (current_graph - np.mean(graph_valid)) / (np.std(graph_valid) + 1e-8)
        if len(mamba_valid) > 0:
            current_mamba = mamba_col[pos]
            features['mamba_current_rank'] = np.mean(current_mamba > mamba_valid) if len(mamba_valid) > 0 else 0.5
            features['mamba_current_zscore'] = (current_mamba - np.mean(mamba_valid)) / (np.std(mamba_valid) + 1e-8)
        if len(transformer_valid) > 0:
            current_transformer = transformer_col[pos]
            features['transformer_current_rank'] = np.mean(current_transformer > transformer_valid) if len(transformer_valid) > 0 else 0.5
            features['transformer_current_zscore'] = (current_transformer - np.mean(transformer_valid)) / (np.std(transformer_valid) + 1e-8)
        return features
    
    def _extract_correlation_features(self, graph_data, mamba_data, transformer_data):
        features = {}
        if len(graph_data) == len(mamba_data):
            corr_graph_mamba = np.corrcoef(graph_data, mamba_data)[0, 1] if len(graph_data) > 1 else 0
            features['corr_graph_mamba'] = 0 if np.isnan(corr_graph_mamba) else corr_graph_mamba
        else:
            features['corr_graph_mamba'] = 0
        if len(graph_data) == len(transformer_data):
            corr_graph_transformer = np.corrcoef(graph_data, transformer_data)[0, 1] if len(graph_data) > 1 else 0
            features['corr_graph_transformer'] = 0 if np.isnan(corr_graph_transformer) else corr_graph_transformer
        else:
            features['corr_graph_transformer'] = 0
        if len(mamba_data) == len(transformer_data):
            corr_mamba_transformer = np.corrcoef(mamba_data, transformer_data)[0, 1] if len(mamba_data) > 1 else 0
            features['corr_mamba_transformer'] = 0 if np.isnan(corr_mamba_transformer) else corr_mamba_transformer
        else:
            features['corr_mamba_transformer'] = 0
        return features

def load_clean_data(file_paths, dataset_name):
    data_dict = {}
    for model_name, path in file_paths.items():
        try:
            df = pd.read_csv(path, header=0, index_col=0)
            df = df.iloc[1:, 1:]
            data_cols = df.columns
            if not data_cols.tolist():
                raise ValueError(f"{model_name} has no data columns after skipping first row/column")
            df_data = df.astype(np.float32)
            if df_data.columns.dtype == 'object' and isinstance(df_data.columns[0], tuple):
                df_data.columns = [''.join(col) for col in df_data.columns]
            data_dict[model_name] = df_data
        except Exception as e:
            raise RuntimeError(f"Failed to load {model_name} for {dataset_name}: {str(e)}")
    all_shapes = [df.shape for df in data_dict.values()]
    if len(set(all_shapes)) != 1:
        raise ValueError(f"All data shapes must be consistent for {dataset_name}! Actual: {all_shapes}")
    true_vals = data_dict["true"].dropna().values
    if not np.all(np.isin(true_vals, [0.0, 1.0])):
        raise ValueError(f"y_true contains non-binary values for {dataset_name}: {np.unique(true_vals)}")
    return data_dict

def prepare_samples(data_dict, dataset_name):
    extractor = AdvancedNeighborhoodFeatureExtractor(window_size=WINDOW_SIZE)
    samples_df = extractor.extract_features_for_dataset(data_dict)
    return samples_df

def _fit_lgbm_classifier(clf, X, y):
    """Fit LightGBM while remaining compatible with old/new LightGBM APIs."""
    try:
        clf.fit(X, y, verbose=50)
    except TypeError:
        clf.fit(X, y)
    return clf


def _build_leakage_safe_cv_splits(X, y, groups, n_splits=5):
    """Create out-of-fold splits using columns as groups whenever possible."""
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)
    class_counts = np.bincount(y, minlength=2)
    if np.count_nonzero(class_counts) < 2:
        return [], "single-class (fixed threshold)"

    max_by_class = int(class_counts[class_counts > 0].min())
    unique_groups = np.unique(groups)
    requested_splits = max(2, min(int(n_splits), max_by_class))

    if StratifiedGroupKFold is not None and len(unique_groups) >= requested_splits:
        try:
            try:
                splitter = StratifiedGroupKFold(
                    n_splits=requested_splits, shuffle=True, random_state=42
                )
            except TypeError:
                splitter = StratifiedGroupKFold(n_splits=requested_splits)
            splits = list(splitter.split(X, y, groups))
            if all(np.unique(y[train_idx]).size == 2 for train_idx, _ in splits):
                return splits, f"StratifiedGroupKFold({requested_splits}) by col_idx"
        except ValueError:
            pass

    group_splits = min(requested_splits, len(unique_groups))
    if group_splits >= 2:
        try:
            splits = list(GroupKFold(n_splits=group_splits).split(X, y, groups))
            if all(np.unique(y[train_idx]).size == 2 for train_idx, _ in splits):
                return splits, f"GroupKFold({group_splits}) by col_idx"
        except ValueError:
            pass

    sample_splits = min(requested_splits, max_by_class)
    if sample_splits >= 2:
        splitter = StratifiedKFold(
            n_splits=sample_splits, shuffle=True, random_state=42
        )
        return list(splitter.split(X, y)), f"StratifiedKFold({sample_splits}) fallback"

    return [], "no valid CV split (fixed threshold)"


def _score_at_threshold(y_true, y_score, threshold):
    """Return Accuracy, F1 and MCC for one fixed threshold."""
    y_true = np.asarray(y_true, dtype=np.int8)
    y_score = np.asarray(y_score, dtype=np.float64)
    y_pred = (y_score >= float(threshold)).astype(np.int8)

    tp = float(np.sum((y_true == 1) & (y_pred == 1)))
    tn = float(np.sum((y_true == 0) & (y_pred == 0)))
    fp = float(np.sum((y_true == 0) & (y_pred == 1)))
    fn = float(np.sum((y_true == 1) & (y_pred == 0)))

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    f1_denom = 2 * tp + fp + fn
    f1 = (2 * tp) / f1_denom if f1_denom else 0.0
    mcc_denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / mcc_denom) if mcc_denom else 0.0
    return float(accuracy), float(f1), float(mcc)


def find_optimal_threshold(y_true, y_score):
    """
    Calibrate a decision threshold on training/calibration predictions only.

    The search is exact over score breakpoints (plus 0.05/0.50/0.95), rather
    than a coarse 61-point grid.  Among thresholds whose accuracy is within
    0.5 percentage points of the best accuracy, it selects the highest F1;
    MCC and proximity to 0.5 are deterministic tie-breakers.

    IMPORTANT: calculate_metrics() never calls this function.  The final test
    set therefore cannot influence the chosen threshold.
    """
    y_true = np.asarray(y_true, dtype=np.int8)
    y_score = np.asarray(y_score, dtype=np.float64)
    valid_mask = np.isfinite(y_score) & np.isin(y_true, [0, 1])
    y_true = y_true[valid_mask]
    y_score = y_score[valid_mask]

    if y_true.size == 0:
        return float(THRESHOLD), 0.0, 0.0
    if np.unique(y_true).size < 2:
        accuracy, f1, _ = _score_at_threshold(y_true, y_score, THRESHOLD)
        return float(THRESHOLD), f1, accuracy

    score_breakpoints = np.unique(y_score[(y_score >= 0.05) & (y_score <= 0.95)])
    thresholds = np.unique(
        np.concatenate((score_breakpoints, np.array([0.05, THRESHOLD, 0.95])))
    )

    order = np.argsort(y_score, kind="mergesort")
    sorted_scores = y_score[order]
    sorted_y = y_true[order]
    suffix_positive = np.r_[np.cumsum(sorted_y[::-1], dtype=np.int64)[::-1], 0]

    first_positive_idx = np.searchsorted(sorted_scores, thresholds, side="left")
    tp = suffix_positive[first_positive_idx].astype(np.float64)
    predicted_positive = (len(y_true) - first_positive_idx).astype(np.float64)
    fp = predicted_positive - tp
    total_positive = float(np.sum(y_true == 1))
    total_negative = float(np.sum(y_true == 0))
    fn = total_positive - tp
    tn = total_negative - fp

    accuracy = (tp + tn) / len(y_true)
    f1_denom = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, f1_denom, out=np.zeros_like(tp), where=f1_denom > 0)
    mcc_denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = np.divide(
        tp * tn - fp * fn,
        mcc_denom,
        out=np.zeros_like(tp),
        where=mcc_denom > 0,
    )

    max_accuracy = float(np.max(accuracy))
    eligible = np.flatnonzero(accuracy >= max_accuracy - 0.005)
    best_idx = max(
        eligible.tolist(),
        key=lambda i: (
            float(f1[i]),
            float(accuracy[i]),
            float(mcc[i]),
            -abs(float(thresholds[i]) - float(THRESHOLD)),
        ),
    )
    return (
        float(thresholds[best_idx]),
        float(f1[best_idx]),
        float(accuracy[best_idx]),
    )


def train_ensemble_classifier(train_samples_df):
    exclude_cols = ['col_idx', 'pos_idx', 'y_true']
    feature_cols = [col for col in train_samples_df.columns if col not in exclude_cols]
    X_train = train_samples_df[feature_cols].values
    y_train = train_samples_df["y_true"].values.astype(int)
    if len(X_train) == 0:
        raise ValueError("No valid training samples, cannot train classifier")

    groups = (
        train_samples_df["col_idx"].values
        if "col_idx" in train_samples_df.columns
        else np.arange(len(train_samples_df))
    )
    cv_splits, cv_method = _build_leakage_safe_cv_splits(
        X_train, y_train, groups, n_splits=5
    )

    decision_threshold = float(THRESHOLD)
    calibration_f1 = np.nan
    calibration_accuracy = np.nan
    calibration_mcc = np.nan

    if cv_splits:
        oof_proba = np.full(len(y_train), np.nan, dtype=np.float64)
        for fold_idx, (fit_idx, valid_idx) in enumerate(cv_splits, start=1):
            print(
                f"Threshold calibration fold {fold_idx}/{len(cv_splits)} "
                f"({cv_method})"
            )
            fold_clf = LGBMClassifier(**ENSEMBLE_PARAMS)
            _fit_lgbm_classifier(fold_clf, X_train[fit_idx], y_train[fit_idx])
            oof_proba[valid_idx] = fold_clf.predict_proba(X_train[valid_idx])[:, 1]

        if np.all(np.isfinite(oof_proba)):
            decision_threshold, calibration_f1, calibration_accuracy = (
                find_optimal_threshold(y_train, oof_proba)
            )
            _, _, calibration_mcc = _score_at_threshold(
                y_train, oof_proba, decision_threshold
            )
        else:
            print(
                "Warning: incomplete OOF predictions; using the fixed 0.5 "
                "threshold instead of consulting any test labels."
            )
    else:
        print(
            "Warning: unable to create leakage-safe OOF folds; using fixed "
            "threshold 0.5."
        )

    print(
        "Leakage-safe ensemble threshold (calibrated on part2 OOF only): "
        f"{decision_threshold:.6f}; F1={calibration_f1:.4f}, "
        f"Accuracy={calibration_accuracy:.4f}, MCC={calibration_mcc:.4f}"
    )

    clf = LGBMClassifier(**ENSEMBLE_PARAMS)
    _fit_lgbm_classifier(clf, X_train, y_train)
    feature_info = {
        'feature_columns': feature_cols,
        'classifier': clf,
        'feature_importance': dict(zip(feature_cols, clf.feature_importances_)),
        'decision_threshold': float(decision_threshold),
        'threshold_calibration': {
            'source': 'part2 training out-of-fold predictions only',
            'cv_method': cv_method,
            'folds': len(cv_splits),
            'f1': float(calibration_f1) if np.isfinite(calibration_f1) else None,
            'accuracy': float(calibration_accuracy) if np.isfinite(calibration_accuracy) else None,
            'mcc': float(calibration_mcc) if np.isfinite(calibration_mcc) else None,
        },
    }
    clf_save_path = f"{output_dir}/ensemble_classifier_balanced_f1_acc.pkl"
    joblib.dump(feature_info, clf_save_path)
    return feature_info

def compute_single_model_metrics(y_true, y_pred_binary, y_score):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred_binary, labels=[0, 1]
    ).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auc = roc_auc_score(y_true, y_score) if np.unique(y_true).size == 2 else np.nan
    metrics = {
        "acc": accuracy_score(y_true, y_pred_binary),
        "mcc": matthews_corrcoef(y_true, y_pred_binary),
        "tpr": tpr,
        "tnr": tnr,
        "f1": f1_score(y_true, y_pred_binary, zero_division=0),
        "auc": auc
    }
    return metrics

def calculate_metrics(samples_df, ensemble_clf_info=None, dataset_name=""):
    metrics_dict = {}
    y_true = samples_df["y_true"].values
    for model_name in ["graph", "mamba", "transformer", "avg", "weighted_avg"]:
        if model_name == "avg":
            y_score = samples_df["avg_score"].values
        elif model_name == "weighted_avg":
            y_score = samples_df["weighted_avg"].values
        else:
            y_score = samples_df[f"{model_name}_current"].values
        y_pred_binary = (y_score >= THRESHOLD).astype(int)
        metrics = compute_single_model_metrics(y_true, y_pred_binary, y_score)
        metrics_dict[model_name] = metrics

    if ensemble_clf_info is not None:
        feature_cols = ensemble_clf_info['feature_columns']
        clf = ensemble_clf_info['classifier']
        X_data = samples_df[feature_cols].values
        y_pred_ensemble_proba = clf.predict_proba(X_data)[:, 1]

        # Critical leakage fix: use the threshold saved during part2-only OOF
        # calibration.  y_true from this dataset is used only for reporting.
        ensemble_threshold = float(
            ensemble_clf_info.get('decision_threshold', THRESHOLD)
        )
        y_pred_ensemble_binary = (
            y_pred_ensemble_proba >= ensemble_threshold
        ).astype(int)
        metrics_ensemble = compute_single_model_metrics(
            y_true, y_pred_ensemble_binary, y_pred_ensemble_proba
        )
        metrics_dict["ensemble"] = metrics_ensemble
        samples_df["ensemble_score"] = y_pred_ensemble_proba
        samples_df["ensemble_pred"] = y_pred_ensemble_binary
        samples_df["ensemble_best_thresh"] = ensemble_threshold
        print(
            f"{dataset_name}: ensemble metrics use fixed training-calibrated "
            f"threshold={ensemble_threshold:.6f}; no threshold search on this set."
        )

    metrics_df = pd.DataFrame(metrics_dict).T
    metrics_df.columns = ["Accuracy", "MCC", "Sensitivity", "Specificity", "F1-Score","AUC"]
    metrics_df.index = [
        "GraphCpG", 
        "MambaCpG", 
        "CpGTransformer", 
        "Average of Three Models",
        "Weighted Average of Three Models",
        "Ensemble Classifier (Balanced F1 & Acc)"
    ]
    metrics_df = metrics_df.round(4)
    return metrics_df, samples_df

def plot_roc_comparison(samples_df, save_dir, dataset_name):
    plt.figure(figsize=(10, 8))
    model_config = {
        "graph": ("GraphCpG", "#1f77b4"),
        "mamba": ("MambaCpG", "#ff7f0e"),
        "transformer": ("CpGTransformer", "#2ca02c"),
        "avg": ("Average of Three Models", "#9467bd"),
        "weighted_avg": ("Weighted Average", "#8c564b"),
        "ensemble": ("Ensemble Classifier (Balanced F1 & Acc)", "#d62728")
    }
    for model_key, (model_label, color) in model_config.items():
        y_true = samples_df["y_true"].values
        if model_key == "ensemble":
            if "ensemble_score" in samples_df.columns:
                y_score = samples_df["ensemble_score"].values
            else:
                continue
        elif model_key == "avg":
            y_score = samples_df["avg_score"].values
        elif model_key == "weighted_avg":
            y_score = samples_df["weighted_avg"].values
        else:
            y_score = samples_df[f"{model_key}_current"].values
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        plt.plot(fpr, tpr, color=color, lw=2,label=f"{model_label} (AUC = {auc:.4f})")
    plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random Guess")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate (FPR)", fontsize=12)
    plt.ylabel("True Positive Rate (TPR)", fontsize=12)
    plt.title(f"ROC Curve Comparison - {dataset_name}", fontsize=14, fontweight="bold", pad=20)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    save_path = f"{save_dir}/roc_comparison_{dataset_name.lower().replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

def analyze_feature_importance(ensemble_clf_info, save_dir, top_n=30):
    feature_importance = ensemble_clf_info['feature_importance']
    feature_cols = ensemble_clf_info['feature_columns']
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': [feature_importance[col] for col in feature_cols]
    }).sort_values('importance', ascending=False)
    plt.figure(figsize=(12, 10))
    top_features = importance_df.head(top_n)
    colors = plt.cm.plasma(np.linspace(0, 1, len(top_features)))
    bars = plt.barh(range(len(top_features)), top_features['importance'], color=colors)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance Score', fontsize=12)
    plt.title(f'Top {top_n} Feature Importance (Balanced F1 & Acc)', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    save_path = f"{save_dir}/feature_importance_balanced.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    importance_path = f"{save_dir}/feature_importance_balanced.csv"
    importance_df.to_csv(importance_path, index=False)
    feature_categories = {
        'Enhanced Base Scores': ['graph_current', 'mamba_current', 'transformer_current', 
                                'weighted_avg', 'max_score', 'min_score', 'score_range', 'score_std'],
        'Statistical': ['mean', 'std', 'median', 'range', 'iqr', 'skew', 'kurtosis', 'q10', 'q25', 'q75', 'q90'],
        'Shape': ['trend', 'peak'],
        'Positional': ['position', 'distance', 'rank', 'zscore'],
        'Correlation': ['corr']
    }
    category_importance = {}
    for category, keywords in feature_categories.items():
        category_features = [col for col in feature_cols if any(keyword in col for keyword in keywords)]
        category_importance[category] = sum(feature_importance[col] for col in category_features if col in feature_importance)
    plt.figure(figsize=(10, 6))
    categories = list(category_importance.keys())
    importances = [category_importance[cat] for cat in categories]
    plt.bar(categories, importances, color=plt.cm.Set3(np.linspace(0, 1, len(categories))))
    plt.xlabel('Feature Category', fontsize=12)
    plt.ylabel('Total Importance Score', fontsize=12)
    plt.title('Feature Importance by Category', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    category_path = f"{save_dir}/feature_importance_by_category.png"
    plt.savefig(category_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("\nTop 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"   {i+1:2d}. {row['feature']}: {row['importance']:.4f}")
    print(f"\nFeature importance by category:")
    for category, importance in category_importance.items():
        print(f"   - {category}: {importance:.4f}")

def create_comparison_table(metrics_train, metrics_test):
    comparison_data = {}
    for model in metrics_train.index:
        comparison_data[f"{model} (part2 - Train)"] = metrics_train.loc[model]
    for model in metrics_test.index:
        comparison_data[f"{model} (part1 - Test)"] = metrics_test.loc[model]
    comparison_df = pd.DataFrame(comparison_data).T
    comparison_df = comparison_df.round(4)
    return comparison_df

def main():
    try:
        print("Cross-Part Analysis: Balance F1 and Accuracy")
        print("Train on part2, Test on part1 (LightGBM + Balanced Metrics)")
        
        print("\n1. Loading Training Data (part2)")
        data_train = load_clean_data(file_paths_train, "part2 (Training)")
        
        print("\n2. Loading Test Data (part1)")
        data_test = load_clean_data(file_paths_test, "part1 (Test)")
        
        print("\n3. Preparing Samples with Enhanced Features")
        train_samples_df = prepare_samples(data_train, "part2 (Training)")
        test_samples_df = prepare_samples(data_test, "part1 (Test)")
        
        print("\n4. Training Ensemble Classifier")
        ensemble_clf_info = train_ensemble_classifier(train_samples_df)
        
        print("\n5. Calculating Metrics for Training Data (part2)")
        metrics_train, train_samples_with_pred = calculate_metrics(
            train_samples_df, ensemble_clf_info, "part2 (Training)"
        )
        
        print("\n6. Calculating Metrics for Test Data (part1)")
        metrics_test, test_samples_with_pred = calculate_metrics(
            test_samples_df, ensemble_clf_info, "part1 (Test)"
        )
        
        print("\nTraining Set Metrics (part2):")
        print(metrics_train.to_string())
        
        print("\nTest Set Metrics (part1):")
        print(metrics_test.to_string())
        
        analyze_feature_importance(ensemble_clf_info, output_dir)
        
        comparison_df = create_comparison_table(metrics_train, metrics_test)
        print("\nCross-Part Comparison:")
        print(comparison_df.to_string())
        
        print("\n7. Saving Results to ./meta_results")
        train_result_path = f"{output_dir}/part2_training_detailed_results_balanced.csv"
        test_result_path = f"{output_dir}/part1_test_detailed_results_balanced.csv"
        train_samples_with_pred.to_csv(train_result_path, index=False)
        test_samples_with_pred.to_csv(test_result_path, index=False)
        metrics_train.to_csv(f"{output_dir}/part2_training_metrics_balanced.csv")
        metrics_test.to_csv(f"{output_dir}/part1_test_metrics_balanced.csv")
        comparison_df.to_csv(f"{output_dir}/cross_part_comparison_balanced.csv")
        
        plot_roc_comparison(train_samples_with_pred, output_dir, "part2 (Training)")
        plot_roc_comparison(test_samples_with_pred, output_dir, "part1 (Test)")
        
        print("\nBalanced F1 & Accuracy Analysis Completed!")
        print("\nSummary:")
        print(f"   - Training samples: {len(train_samples_df)}")
        print(f"   - Test samples: {len(test_samples_df)}")
        print(f"   - Total features: {len(ensemble_clf_info['feature_columns'])}")
        print(f"   - Fixed ensemble threshold (part2 OOF): {ensemble_clf_info['decision_threshold']:.6f}")
        print(f"   - Results saved to: {os.path.abspath(output_dir)}")
        
    except Exception as e:
        print(f"\nProcess interrupted: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main()
    

Cross-Part Analysis: Balance F1 and Accuracy
Train on part2, Test on part1 (LightGBM + Balanced Metrics)

1. Loading Training Data (part2)

2. Loading Test Data (part1)

3. Preparing Samples with Enhanced Features

4. Training Ensemble Classifier
Threshold calibration fold 1/5 (StratifiedGroupKFold(5) by col_idx)
[LightGBM] [Info] Number of positive: 25950, number of negative: 38696
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.870446
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.491991
[LightGBM] [Debug] init for col-wise cost 0.996016 seconds, init for row-wise cost 0.901557 seconds
[LightGBM] [Debug] col-wise cost 0.056096 seconds, row-wise cost 0.045999 seconds
[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 1.098111 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin


[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree 

[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 39 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 37 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 39 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debu

[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree 

[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree 

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 38 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 31 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 22 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 29 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM]

[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree 

[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 19 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 37 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 40 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 20 and max_depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]


Balanced F1 & Accuracy Analysis Completed!

Summary:
   - Training samples: 78369
   - Test samples: 86038
   - Total features: 72
   - Fixed ensemble threshold (part2 OOF): 0.394354
   - Results saved to: /data/cabins/methyeval/lianght/cpg-transformer-main/merge_pott/meta_results


In [7]:
import pandas as pd
import numpy as np
import os
import joblib
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def custom_find_peaks(data, height=None, distance=1):
    if len(data) < 3:
        return np.array([], dtype=int), {'peak_heights': np.array([])}
    peaks = [i for i in range(1, len(data)-1) if data[i] > data[i-1] and data[i] > data[i+1]]
    peaks = np.array(peaks, dtype=int)
    if height is not None:
        peak_heights = data[peaks]
        peaks = peaks[peak_heights >= height]
    if distance > 1 and len(peaks) > 1:
        filtered_peaks = [peaks[0]]
        for p in peaks[1:]:
            if p - filtered_peaks[-1] >= distance:
                filtered_peaks.append(p)
        peaks = np.array(filtered_peaks)
    return peaks, {'peak_heights': data[peaks] if len(peaks) > 0 else np.array([])}

def fill_window_nan(window_data):
    win_mean = np.nanmean(window_data) if np.sum(~np.isnan(window_data)) > 0 else 0.0
    return np.where(np.isnan(window_data), win_mean, window_data)

class FeatureExtractor:
    def __init__(self, window_size=10):
        self.window_size = window_size

    def extract(self, data_dict, debug=False):
        ensemble = data_dict["ensemble"].astype(np.float32)
        true = data_dict["true"].astype(np.float32)

        if debug:
            total_true_nan = true.isna().sum().sum()
            print(f"Debug: Original y_true NaNs in unimputed: {total_true_nan}")
            print(f"Debug: First 5 rows of unimputed y_true:\n{true.iloc[:5, :5]}")

        n_positions, n_cols = ensemble.shape
        all_features = []

        for col_idx in tqdm(range(n_cols), desc="Processing columns", unit="col"):
            e_col = ensemble.iloc[:, col_idx].values
            y_col = true.iloc[:, col_idx].values

            for pos_idx in tqdm(range(n_positions), desc=f"Column {col_idx}", unit="pos", leave=False):
                if np.isnan(e_col[pos_idx]):
                    continue

                e_val = e_col[pos_idx]

                base = {
                    'ensemble_current': e_val,
                    'ensemble_abs': abs(e_val),
                    'col_idx': col_idx, 
                    'pos_idx': pos_idx,
                    'y_true': y_col[pos_idx]
                }

                window_start = max(0, pos_idx - self.window_size)
                window_end = min(n_positions, pos_idx + self.window_size + 1)
                e_win = fill_window_nan(e_col[window_start:window_end])

                adv = {}
                adv.update(self._stat_features(e_win, 'ensemble'))
                adv.update(self._shape_features(e_win, 'ensemble'))
                adv.update(self._pos_features(e_win, pos_idx, window_start, window_end))

                all_features.append({**base, **adv})
        
        features_df = pd.DataFrame(all_features)
        if not features_df.empty:
            feature_cols = [col for col in features_df.columns if col != 'y_true']
            features_df[feature_cols] = features_df[feature_cols].fillna(0.0)

        if debug:
            extracted_nan = features_df['y_true'].isna().sum() if not features_df.empty else 0
            print(f"Debug: Extracted features y_true NaNs: {extracted_nan}")
            if len(features_df) > 0:
                print(f"Debug: First 5 samples y_true in features: {features_df['y_true'].head(5).values}")

        print(f"Extracted features: total samples={len(features_df)}, y_true NaN samples={features_df['y_true'].isna().sum() if not features_df.empty else 0}")
        return features_df

    def _stat_features(self, data, prefix):
        return {
            f'{prefix}_neighbor_mean': np.mean(data),
            f'{prefix}_neighbor_std': np.std(data) if len(data) > 1 else 0.0,
            f'{prefix}_neighbor_median': np.median(data),
            f'{prefix}_neighbor_range': np.ptp(data) if len(data) > 1 else 0.0,
            f'{prefix}_neighbor_iqr': stats.iqr(data) if len(data) > 4 else 0.0,
            f'{prefix}_neighbor_skew': stats.skew(data) if len(data) > 2 else 0.0,
            f'{prefix}_neighbor_kurtosis': stats.kurtosis(data) if len(data) > 3 else 0.0,
            f'{prefix}_neighbor_valid_count': len(data),
            f'{prefix}_neighbor_q10': np.quantile(data, 0.1),
            f'{prefix}_neighbor_q25': np.quantile(data, 0.25),
            f'{prefix}_neighbor_q75': np.quantile(data, 0.75),
            f'{prefix}_neighbor_q90': np.quantile(data, 0.9)
        }

    def _shape_features(self, data, prefix):
        if len(data) <= 2:
            return {
                f'{prefix}_neighbor_trend_slope': 0.0,
                f'{prefix}_neighbor_trend_r2': 0.0,
                f'{prefix}_neighbor_trend_pvalue': 1.0,
                f'{prefix}_neighbor_peak_count': 0,
                f'{prefix}_neighbor_peak_mean_height': 0.0
            }
        slope, _, r_val, p_val, _ = stats.linregress(np.arange(len(data)), data)
        peaks, props = custom_find_peaks(data, height=np.mean(data), distance=2)
        return {
            f'{prefix}_neighbor_trend_slope': slope,
            f'{prefix}_neighbor_trend_r2': r_val**2,
            f'{prefix}_neighbor_trend_pvalue': p_val,
            f'{prefix}_neighbor_peak_count': len(peaks),
            f'{prefix}_neighbor_peak_mean_height': np.mean(props['peak_heights']) if len(peaks) > 0 else 0.0
        }

    def _pos_features(self, e_win, pos_idx, win_start, win_end):
        window_center = (win_start + win_end - 1) / 2
        return {
            'position_relative_to_center': pos_idx - window_center,
            'distance_to_start': pos_idx - win_start,
            'distance_to_end': win_end - 1 - pos_idx,
            'ensemble_current_rank': np.mean(e_win[pos_idx - win_start] > e_win),
            'ensemble_current_zscore': (e_win[pos_idx - win_start] - np.mean(e_win)) / (np.std(e_win) + 1e-8)
        }

def load_data(file_paths, dataset_name, debug=False):
    data_dict = {}
    for model_name, path in file_paths.items():
        df = pd.read_csv(
            path, 
            header=0, 
            index_col=0,
            na_values=['', 'NA', 'na', 'NaN', 'nan'],
            keep_default_na=True
        ).iloc[1:, 1:]

        if model_name == "true":
            df = df.astype(str).replace('NaN', np.nan)
            df = df.apply(pd.to_numeric, errors='coerce')

        df = df.astype(np.float32)
        data_dict[model_name] = df
        print(f"Loaded {model_name}: shape (positions={df.shape[0]}, columns={df.shape[1]})")

        if debug and model_name == "true":
            total_nan = df.isna().sum().sum()
            print(f"Debug: {dataset_name} y_true NaNs after loading: {total_nan}")
            print(f"Debug: First 5 rows of {dataset_name} y_true:\n{df.iloc[:5, :5]}")
    
    return data_dict

def impute_unimputed_with_ensemble(model_path, unimputed_data, trained_data):
    model_info = joblib.load(model_path)
    # Preserve the exact feature order used by LightGBM.  Converting this list
    # to a set (the old code) could silently permute columns and hurt results.
    model_feats = list(model_info['feature_columns'])
    model_feat_set = set(model_feats)
    print(f"Loaded ensemble model: {len(model_feats)} features")

    extractor = FeatureExtractor()
    print("Extracting features from unimputed data...")
    unimputed_features = extractor.extract(unimputed_data, debug=True)

    # `trained_data` is kept in the function signature for compatibility, but
    # its labels are deliberately not inspected.  The old implementation used
    # part1 labels to re-select the threshold, which leaked the held-out test
    # truth.  The threshold now comes from part2-only OOF calibration saved in
    # the model artifact.
    missing_feats = model_feat_set - set(unimputed_features.columns)
    if missing_feats:
        print(f"Warning: Missing features (fill with 0): {missing_feats}")
        for feat in missing_feats:
            unimputed_features[feat] = 0.0

    unimputed_to_impute = unimputed_features[
        unimputed_features['y_true'].isna()
    ][model_feats + ['col_idx', 'pos_idx']]
    print(f"Found {len(unimputed_to_impute)} samples to impute")

    best_thresh = float(model_info.get('decision_threshold', 0.5))
    threshold_meta = model_info.get('threshold_calibration', {})
    threshold_source = threshold_meta.get(
        'source', 'fixed 0.5 fallback (legacy model without calibration metadata)'
    )
    print(
        f"Using saved threshold: {best_thresh:.6f}; source: {threshold_source}. "
        "No part1/Test labels are used."
    )

    imputed_true = unimputed_data["true"].copy()
    if len(unimputed_to_impute) == 0:
        print("No missing targets require imputation")
        return imputed_true

    X_impute = unimputed_to_impute[model_feats].values
    y_proba = model_info['classifier'].predict_proba(X_impute)[:, 1]
    y_pred = (y_proba >= best_thresh).astype(int).astype(float)

    impute_count = 0
    for _, row in tqdm(
        unimputed_to_impute.iterrows(),
        total=len(unimputed_to_impute),
        desc="Imputing",
    ):
        col_idx, pos_idx = int(row['col_idx']), int(row['pos_idx'])
        if 0 <= col_idx < imputed_true.shape[1] and 0 <= pos_idx < imputed_true.shape[0]:
            imputed_true.iloc[pos_idx, col_idx] = y_pred[impute_count]
            impute_count += 1

    print(f"Imputed {impute_count}/{len(unimputed_to_impute)} NaNs")
    return imputed_true

def main():
    ENSEMBLE_MODEL_PATH = "./meta_results/ensemble_classifier.pkl"
    TRAINED_DATA_PATHS = {
        "ensemble": os.path.join("processed_data", "ensemble_pred_part1.csv"),
        "true": os.path.join("processed_data", "transformer_y_true_part1.csv")
    }
    UNIMPUTED_DATA_PATHS = {
        "ensemble": os.path.join("processed_data", "ensemble_pred_unimputed.csv"),
        "true": os.path.join("processed_data", "transformer_y_true_unimputed.csv")
    }
    OUTPUT_PATH = "./imputed_data/transformer_y_true_imputed.csv"

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    print("="*60)
    print("Imputing with Ensemble Model Only")
    print("="*60)

    print("\nStep 1: Reusing the threshold already calibrated on part2 OOF")
    trained_data = None  # Do not read part1/Test labels for threshold selection.

    print("\nStep 2: Loading unimputed data")
    unimputed_data = load_data(UNIMPUTED_DATA_PATHS, "unimputed data", debug=True)

    print("\nStep 3: Imputing with ensemble model")
    imputed_true = impute_unimputed_with_ensemble(
        model_path=ENSEMBLE_MODEL_PATH,
        unimputed_data=unimputed_data,
        trained_data=trained_data
    )

    print("\nStep 4: Saving results")
    imputed_true.to_csv(OUTPUT_PATH, index=True)
    print(f"Saved to: {os.path.abspath(OUTPUT_PATH)}")

    original_nan = unimputed_data["true"].isna().sum().sum()
    remaining_nan = imputed_true.isna().sum().sum()
    print(f"\nOriginal NaNs: {original_nan}, Imputed: {original_nan - remaining_nan}")

if __name__ == "__main__":
    main()


Imputing with Ensemble Model Only

Step 1: Reusing the threshold already calibrated on part2 OOF

Step 2: Loading unimputed data


FileNotFoundError: [Errno 2] No such file or directory: 'processed_data/ensemble_pred_unimputed.csv'

In [1]:
# ============================================================
# Final CSV ordered-alignment audit
# Only the final CSV files in ./processed_data are inspected.
# No NPZ file is read in this cell.
# ============================================================
import os
import gc
import hashlib
import numpy as np
import pandas as pd

DATA_DIR = "./processed_data"

# These are exactly the final CSV files used by the ensemble code.
CSV_FILES = {
    "part1": {
        "graph": "graphCpG_part1.csv",
        "mamba_pred": "mamba_y_pred_prob_part1.csv",
        "transformer_pred": "transformer_y_pred_prob_part1.csv",
        "mamba_true": "mamba_y_true_part1.csv",
        "transformer_true": "transformer_y_true_part1.csv",
    },
    "part2": {
        "graph": "graphCpG_part2.csv",
        "mamba_pred": "mamba_y_pred_prob_part2.csv",
        "transformer_pred": "transformer_y_pred_prob_part2.csv",
        "mamba_true": "mamba_y_true_part2.csv",
        "transformer_true": "transformer_y_true_part2.csv",
    },
}

# Match the final ensemble loader in the current notebook:
# pd.read_csv(..., index_col=0), followed by df.iloc[1:, 1:].
SKIP_FIRST_DATA_ROW = True
SKIP_FIRST_DATA_COLUMN = True


def read_final_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Final CSV not found: {path}")

    df = pd.read_csv(path, header=0, index_col=0)

    # Normalize labels only for reliable comparison; row order is never changed.
    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    if df.index.has_duplicates:
        duplicated = df.index[df.index.duplicated()].tolist()[:5]
        raise AssertionError(
            f"{os.path.basename(path)} contains duplicated row labels: {duplicated}"
        )
    if df.columns.has_duplicates:
        duplicated = df.columns[df.columns.duplicated()].tolist()[:5]
        raise AssertionError(
            f"{os.path.basename(path)} contains duplicated column labels: {duplicated}"
        )

    return df


def first_sequence_difference(left, right):
    limit = min(len(left), len(right))
    for i in range(limit):
        if left[i] != right[i]:
            return i, left[i], right[i]
    if len(left) != len(right):
        left_value = left[limit] if limit < len(left) else "<END>"
        right_value = right[limit] if limit < len(right) else "<END>"
        return limit, left_value, right_value
    return None


def assert_same_ordered_layout(reference, candidate, reference_name, candidate_name):
    if reference.shape != candidate.shape:
        raise AssertionError(
            f"{candidate_name} and {reference_name} have different shapes: "
            f"{candidate.shape} vs {reference.shape}"
        )

    row_difference = first_sequence_difference(
        reference.index.tolist(), candidate.index.tolist()
    )
    if row_difference is not None:
        position, ref_value, candidate_value = row_difference
        raise AssertionError(
            f"Row-order mismatch between {reference_name} and {candidate_name} "
            f"at row position {position}: {ref_value!r} vs {candidate_value!r}"
        )

    column_difference = first_sequence_difference(
        reference.columns.tolist(), candidate.columns.tolist()
    )
    if column_difference is not None:
        position, ref_value, candidate_value = column_difference
        raise AssertionError(
            f"Column-order mismatch between {reference_name} and {candidate_name} "
            f"at column position {position}: {ref_value!r} vs {candidate_value!r}"
        )


def first_value_difference(left, right):
    left_array = left.to_numpy(dtype=np.float64)
    right_array = right.to_numpy(dtype=np.float64)

    equal_mask = np.isclose(
        left_array,
        right_array,
        rtol=0.0,
        atol=0.0,
        equal_nan=True,
    )
    if np.all(equal_mask):
        return None

    row_position, column_position = np.argwhere(~equal_mask)[0]
    return {
        "row_position": int(row_position),
        "column_position": int(column_position),
        "row_label": left.index[row_position],
        "column_label": left.columns[column_position],
        "left_value": left_array[row_position, column_position],
        "right_value": right_array[row_position, column_position],
    }


def ordered_ground_truth_digest(df):
    """Fingerprint that changes if shape, labels, NaN mask, values or order changes."""
    values = df.to_numpy(dtype=np.float64)
    nan_mask = np.isnan(values)
    normalized = np.nan_to_num(
        values,
        nan=9.87654321e307,
        posinf=8.76543210e307,
        neginf=-8.76543210e307,
    )

    digest = hashlib.sha256()
    digest.update(str(df.shape).encode("utf-8"))
    digest.update("\n".join(df.index.tolist()).encode("utf-8"))
    digest.update("\n".join(df.columns.tolist()).encode("utf-8"))
    digest.update(nan_mask.tobytes())
    digest.update(np.ascontiguousarray(normalized).tobytes())
    return digest.hexdigest()


def get_actually_used_region(df):
    row_start = 1 if SKIP_FIRST_DATA_ROW else 0
    column_start = 1 if SKIP_FIRST_DATA_COLUMN else 0
    return df.iloc[row_start:, column_start:]


print("=" * 78)
print("FINAL CSV ORDERED-ALIGNMENT AUDIT")
print("Only ./processed_data/*.csv files are being checked.")
print("=" * 78)

audit_rows = []

for part_name, filenames in CSV_FILES.items():
    print(f"\n[{part_name.upper()}] Loading final CSV files")

    dataframes = {
        name: read_final_csv(os.path.join(DATA_DIR, filename))
        for name, filename in filenames.items()
    }

    reference_name = "transformer_true"
    reference = dataframes[reference_name]

    # 1. Check raw final CSV layout without sorting, merging or reindexing.
    for name, dataframe in dataframes.items():
        assert_same_ordered_layout(
            reference,
            dataframe,
            reference_name,
            name,
        )

    print(
        f"[PASS] All five raw CSVs have exactly the same shape, "
        f"row-label order and column-label order: {reference.shape}"
    )

    # 2. The two independently exported y_true CSVs must be identical.
    true_difference = first_value_difference(
        dataframes["transformer_true"],
        dataframes["mamba_true"],
    )
    if true_difference is not None:
        raise AssertionError(
            "MambaCpG and CpGTransformer y_true CSVs are not identical. "
            f"First mismatch: row position {true_difference['row_position']} "
            f"({true_difference['row_label']}), column position "
            f"{true_difference['column_position']} "
            f"({true_difference['column_label']}), "
            f"Transformer={true_difference['left_value']}, "
            f"Mamba={true_difference['right_value']}."
        )

    print(
        "[PASS] MambaCpG and CpGTransformer y_true CSVs are identical in "
        "shape, NaN positions, values and element order."
    )

    # 3. Reproduce the exact slicing performed by the final ensemble loader.
    used = {
        name: get_actually_used_region(dataframe)
        for name, dataframe in dataframes.items()
    }
    used_reference = used[reference_name]

    for name, dataframe in used.items():
        assert_same_ordered_layout(
            used_reference,
            dataframe,
            f"{reference_name} used region",
            f"{name} used region",
        )

    used_true_difference = first_value_difference(
        used["transformer_true"],
        used["mamba_true"],
    )
    if used_true_difference is not None:
        raise AssertionError(
            "The y_true matrices differ in the region actually used for training/evaluation."
        )

    print(
        f"[PASS] After applying the same iloc slicing as the ensemble code, "
        f"all model matrices remain identically ordered: {used_reference.shape}"
    )

    # 4. Report position-wise availability without requiring prediction NaN masks
    # to be identical (different methods may legitimately miss different positions).
    true_values = used["transformer_true"].to_numpy(dtype=np.float64)
    graph_values = used["graph"].to_numpy(dtype=np.float64)
    mamba_values = used["mamba_pred"].to_numpy(dtype=np.float64)
    transformer_values = used["transformer_pred"].to_numpy(dtype=np.float64)

    observed_truth = ~np.isnan(true_values)
    graph_available = ~np.isnan(graph_values)
    mamba_available = ~np.isnan(mamba_values)
    transformer_available = ~np.isnan(transformer_values)

    joint_valid = (
        observed_truth
        & graph_available
        & mamba_available
        & transformer_available
    )

    non_binary = true_values[
        observed_truth
        & ~np.isin(true_values, [0.0, 1.0])
    ]
    if non_binary.size:
        raise AssertionError(
            f"{part_name}: y_true contains non-binary observed values, "
            f"for example {np.unique(non_binary)[:10]}"
        )

    digest = ordered_ground_truth_digest(used["transformer_true"])

    print(f"[PASS] Observed y_true values are binary.")
    print(f"       Ordered y_true SHA-256: {digest}")
    print(f"       Observed y_true positions: {int(observed_truth.sum()):,}")
    print(
        f"       GraphCpG available at observed positions: "
        f"{int((observed_truth & graph_available).sum()):,}"
    )
    print(
        f"       MambaCpG available at observed positions: "
        f"{int((observed_truth & mamba_available).sum()):,}"
    )
    print(
        f"       CpGTransformer available at observed positions: "
        f"{int((observed_truth & transformer_available).sum()):,}"
    )
    print(
        f"       Position-wise common samples actually usable by all models: "
        f"{int(joint_valid.sum()):,}"
    )

    audit_rows.append({
        "part": part_name,
        "raw_rows": reference.shape[0],
        "raw_columns": reference.shape[1],
        "used_rows": used_reference.shape[0],
        "used_columns": used_reference.shape[1],
        "observed_y_true": int(observed_truth.sum()),
        "joint_valid_samples": int(joint_valid.sum()),
        "ordered_y_true_sha256": digest,
        "status": "PASS",
    })

    del dataframes, used
    gc.collect()

audit_df = pd.DataFrame(audit_rows)
print("\n" + "=" * 78)
print("FINAL RESULT: PASS")
print(
    "The final CSVs use the same ordered row-column grid, and the independently "
    "exported MambaCpG/CpGTransformer ground-truth matrices are exactly identical."
)
print("No sorting, merging or implicit reindexing was used in this audit.")
print("=" * 78)
display(audit_df)

FINAL CSV ORDERED-ALIGNMENT AUDIT
Only ./processed_data/*.csv files are being checked.

[PART1] Loading final CSV files
[PASS] All five raw CSVs have exactly the same shape, row-label order and column-label order: (206848, 30)
[PASS] MambaCpG and CpGTransformer y_true CSVs are identical in shape, NaN positions, values and element order.
[PASS] After applying the same iloc slicing as the ensemble code, all model matrices remain identically ordered: (206847, 29)
[PASS] Observed y_true values are binary.
       Ordered y_true SHA-256: cecddaab97b104aa27dc3047d9022e6da082727e1fb35a3a4275a0bd5ba97467
       Observed y_true positions: 86,038
       GraphCpG available at observed positions: 86,038
       MambaCpG available at observed positions: 86,038
       CpGTransformer available at observed positions: 86,038
       Position-wise common samples actually usable by all models: 86,038

[PART2] Loading final CSV files
[PASS] All five raw CSVs have exactly the same shape, row-label order and c

,part,raw_rows,raw_columns,used_rows,used_columns,observed_y_true,joint_valid_samples,ordered_y_true_sha256,status
0,part1,206848,30,206847,29,86038,86038,cecddaab97b104aa27dc3047d9022e6da082727e1fb35a...,PASS
1,part2,191488,30,191487,29,78369,78369,b984037ba51e45ef937f5482e5d52e493071089d2b0e98...,PASS
